# Gold — Indicator Dimension
Describes each economic indicator measured in the fact tables.\
Includes descriptive attributes and a foreign key to the source dimension.

**Output:** `gold.dim_indicator` (Delta table)  
**Grain:** One row per economic indicator

In [ ]:
import pandas as pd

# source_key: 1=Seðlabanki Íslands, 2=Hagstofa Íslands
indicators = [
    (1, "policy_rate",    "Central Bank Policy Rate",   "Monetary Policy", "Interest Rate", "Percent (%)",          "End-of-period", None,  1),
    (2, "cpi",            "CPI Year-on-Year Inflation", "Inflation",       "Price Index",   "Percent (%)",          "End-of-period", False, 1),
    (3, "iskeur",         "ISK/EUR Exchange Rate",      "FX",              "Exchange Rate", "ISK per 1 EUR",        "Average",       None,  1),
    (4, "gdp_yoy_growth", "GDP Year-on-Year Growth",    "GDP",             "Growth Rate",   "Percent (%)",          "End-of-period", True,  2),
    (5, "hpi",            "House Price Index",          "Housing",         "Price Index",   "Index (Mar 2000=100)", "End-of-period", False, 2),
    (6, "wage_index",     "Wage Index",                 "Labour Market",   "Price Index",   "Index (Dec 1988=100)", "End-of-period", True,  2),
]

df_ind = pd.DataFrame(indicators, columns=[
    "indicator_key", "indicator_code", "indicator_name",
    "category", "subcategory", "unit", "aggregation_type",
    "is_higher_better", "source_key"
])

In [ ]:
df_spark = spark.createDataFrame(df_ind)
df_spark.createOrReplaceTempView("v_dim_indicator")

if not spark.catalog.tableExists("gold.dim_indicator"):
    df_spark.write.format("delta").saveAsTable("gold.dim_indicator")
else:
    spark.sql("""
        MERGE INTO gold.dim_indicator AS target
        USING v_dim_indicator AS source
        ON target.indicator_key = source.indicator_key
        WHEN MATCHED THEN UPDATE SET
            target.indicator_code  = source.indicator_code,
            target.indicator_name  = source.indicator_name,
            target.category        = source.category,
            target.subcategory     = source.subcategory,
            target.unit            = source.unit,
            target.aggregation_type = source.aggregation_type,
            target.is_higher_better = source.is_higher_better,
            target.source_key      = source.source_key
        WHEN NOT MATCHED THEN INSERT (
            indicator_key, indicator_code, indicator_name, category, subcategory,
            unit, aggregation_type, is_higher_better, source_key
        )
        VALUES (
            source.indicator_key, source.indicator_code, source.indicator_name,
            source.category, source.subcategory, source.unit, source.aggregation_type,
            source.is_higher_better, source.source_key
        )
    """)

print(f"Dimension updated: {spark.table('gold.dim_indicator').count()} rows.")